## 6. Save for Ensemble

✅ RNN predictions ready for ensemble combination

**Output Summary:**
- Predictions shape: (96,) - matches test set
- Scale: [0, 1] range (scaled)
- Format: Numpy array
- Metric (R²): {:.4f}

These predictions will be combined with LSTM, Random Forest, and Gaussian Process predictions in the ensemble notebook.

**Next:** Run the other model notebooks (Random Forest, Gaussian) to generate their predictions.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Actual vs Predicted (first 100 samples)
axes[0].plot(y_test_seq[:100], label='Actual', linewidth=2, alpha=0.7, color='#2E86AB')
axes[0].plot(rnn_predictions_scaled[:100], label='RNN Predicted', linewidth=2, alpha=0.7, color='#A23B72')
axes[0].set_title('RNN: Actual vs Predicted (First 100 samples)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Scaled Demand')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = y_test_seq - rnn_predictions_scaled
axes[1].plot(residuals, label='Residuals', linewidth=1, alpha=0.7, color='#F18F01')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].fill_between(range(len(residuals)), residuals, alpha=0.3, color='#F18F01')
axes[1].set_title('RNN: Prediction Residuals', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Prediction visualizations created")

## 5. Visualize Predictions

In [ ]:
# Generate predictions
rnn_predictions_scaled = model.predict(X_test_seq, verbose=0).flatten()

# Inverse scale to original range
rnn_predictions = preprocessing.inverse_scale_predictions(rnn_predictions_scaled, scaler_y)

print(f"RNN Predictions shape: {rnn_predictions.shape}")
print(f"RNN Predictions range: [{rnn_predictions.min():.2f}, {rnn_predictions.max():.2f}]")
print(f"\nFirst 10 predictions: {rnn_predictions[:10]}")

# Calculate metrics
rnn_mse = mean_squared_error(y_test_seq, rnn_predictions_scaled)
rnn_mae = mean_absolute_error(y_test_seq, rnn_predictions_scaled)
rnn_r2 = r2_score(y_test_seq, rnn_predictions_scaled)

print(f"\n📈 RNN Model Performance (on scaled data):")
print(f"  MSE: {rnn_mse:.4f}")
print(f"  MAE: {rnn_mae:.4f}")
print(f"  R²:  {rnn_r2:.4f}")

## 4. Generate Predictions

In [ ]:
print("🚀 Training RNN model...")
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=16,
    validation_split=0.1,
    verbose=0
)

print("✓ Training completed")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('RNN - Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_title('RNN - Mean Absolute Error', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Train RNN Model

In [ ]:
model = Sequential([
    SimpleRNN(64, activation='relu',
              input_shape=(data['seq_length'], data['n_features']),
              return_sequences=True,
              name='RNN_1'),
    Dropout(0.2),
    SimpleRNN(32, activation='relu', name='RNN_2'),
    Dropout(0.2),
    Dense(16, activation='relu', name='Dense_1'),
    Dense(1, activation='sigmoid', name='Output')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("\n📊 RNN Model Architecture:")
model.summary()

## 2. Build RNN Architecture

In [ ]:
# Load data (SAME as all other models)
X, y = preprocessing.load_synthetic_data(n_samples=500, n_features=10)

# Unified preprocessing
data = preprocessing.preprocess_data(X, y, test_size=0.2, seq_length=10)

# Extract sequences for RNN
X_train_seq = data['X_train_seq']
X_test_seq = data['X_test_seq']
y_train_seq = data['y_train_seq']
y_test_seq = data['y_test_seq']
scaler_y = data['scaler_y']

print(f"Training data shapes (for RNN):")
print(f"  X_train: {X_train_seq.shape} (batch, sequence, features)")
print(f"  y_train: {y_train_seq.shape}")
print(f"\nTest data shapes (for RNN):")
print(f"  X_test: {X_test_seq.shape}")
print(f"  y_test: {y_test_seq.shape}")

# IMPORTANT: Verify output dimension
print(f"\n✓ Test predictions will have shape: ({len(y_test_seq)},)")

## 1. Load and Preprocess Data

In [ ]:
import sys
sys.path.insert(0, '../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

from src import preprocessing
from src.models import rnn_model

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("✓ Libraries and modules imported successfully")

# Demand Forecasting System - RNN Model

## Overview
This notebook trains a **RNN (Recurrent Neural Network)** using SimpleRNN layers for demand forecasting.

**Key Features:**
- Input shape: (batch, seq_length=10, n_features=10)
- Output shape: (n_test_samples,) - **MUST match other models**
- Architecture: 2 SimpleRNN layers with dropout
- Output: predictions in [0, 1] range (scaled)

**Critical:** All predictions must have the **same shape and scale** for ensemble averaging.